In [1]:
import torch
from transformers import GPT2LMHeadModel, DataCollatorForLanguageModeling, TrainingArguments, Trainer, set_seed
import pandas as pd
import numpy as np
from datasets import Dataset, load_from_disk, concatenate_datasets
from transformers import GPT2TokenizerFast
from pathlib import Path
import json

In [2]:
seed = 45
set_seed(seed)
np.random.seed(seed)

In [3]:
out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer = GPT2TokenizerFast.from_pretrained(out_dir / 'tokenizer')

with open(out_dir / 'token_spec.json', 'r', encoding='utf-8') as f:
    token_spec = json.load(f)
    
ds_user_behavior = load_from_disk(out_dir / 'data' / 'behavior')
ds_meta_title = load_from_disk(out_dir / 'data' / 'meta_title')
ds_meta_genre = load_from_disk(out_dir / 'data' / 'meta_genre')
ds_meta_year  = load_from_disk(out_dir / 'data' / 'meta_year')

def sample_len_stats(ds, n=2000):
    m = min(n, len(ds))
    lens = [len(ds[i]['input_ids']) for i in range(m)]
    return {
        'min': int(np.min(lens)),
        'median': int(np.median(lens)),
        'p95': int(np.percentile(lens, 95)),
        'max': int(np.max(lens)),
    }

print('behavior:', sample_len_stats(ds_user_behavior))
print('meta_title:', sample_len_stats(ds_meta_title))
print('meta_genre:', sample_len_stats(ds_meta_genre))
print('meta_year:', sample_len_stats(ds_meta_year))

ratio_behavior = 0.5

ds_meta_all = concatenate_datasets([ds_meta_title, ds_meta_genre, ds_meta_year]).shuffle(seed=seed)
ds_user_behavior = ds_user_behavior.shuffle(seed=seed)

max_total = int(min(len(ds_user_behavior) / ratio_behavior, len(ds_meta_all) / (1 - ratio_behavior)))

n_user_behavior = int(max_total * ratio_behavior)
n_meta_all = int(max_total * (1 - ratio_behavior))

ds_user_behavior_selected = ds_user_behavior.select(range(n_user_behavior))
ds_meta_all_selected = ds_meta_all.select(range(n_meta_all))

ds_cpt = concatenate_datasets([ds_user_behavior_selected, ds_meta_all_selected]).shuffle(seed=seed)

print('ds_cpt size:', len(ds_cpt), 'behavior:', n_user_behavior, 'meta:', n_meta_all)

split = ds_cpt.train_test_split(test_size=0.02, seed=seed)
ds_train = split['train']
ds_val = split['test']

print('train:', len(ds_train), 'val:', len(ds_val))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


behavior: {'min': 152, 'median': 728, 'p95': 1008, 'max': 1008}
meta_title: {'min': 15, 'median': 17, 'p95': 22, 'max': 33}
meta_genre: {'min': 17, 'median': 20, 'p95': 25, 'max': 33}
meta_year: {'min': 16, 'median': 16, 'p95': 16, 'max': 16}
ds_cpt size: 12080 behavior: 6040 meta: 6040
train: 11838 val: 242


In [4]:
model = GPT2LMHeadModel.from_pretrained('gpt2-medium')
model.resize_token_embeddings(len(tokenizer))

model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id 

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

out_model_dir = out_dir / 'cpt_gpt2-medium_v1'

args = TrainingArguments(
    output_dir=str(out_model_dir),
    overwrite_output_dir=False,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, 
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=500,
    save_steps=500,
    logging_steps=50,
    fp16=True,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=1,
    
    
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(out_model_dir))
tokenizer.save_pretrained(str(out_model_dir))

Step,Training Loss,Validation Loss
500,2.576800,2.438909
1000,2.025500,1.926980
1500,1.861400,1.744233
2000,1.769500,1.630922


KeyboardInterrupt: 

In [6]:
tokenizer.save_pretrained(str(out_model_dir))

('..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1\\tokenizer_config.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1\\special_tokens_map.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1\\vocab.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1\\merges.txt',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1\\added_tokens.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1\\tokenizer.json')

In [7]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1')
best_dir = base_dir / 'checkpoint-2000'

tokenizer = GPT2TokenizerFast.from_pretrained(str(base_dir))
model = GPT2LMHeadModel.from_pretrained(str(best_dir))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [8]:
def generate_from_prompt(prompt, max_new_tokens=60, sample=False):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            do_sample=sample,
            top_p=0.9,
            temperature=0.25,
            pad_token_id=model.config.pad_token_id,
            eos_token_id=model.config.eos_token_id,
            bos_token_id=model.config.bos_token_id
        )
        

    return tokenizer.decode(out[0])

In [12]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'title' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><title>Movie <sid_0_739><sid_1_419><sid_2_63><sid_3_153><sid_4_53> has title: Dogma</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_739><sid_1_419><sid_2_63><sid_3_153><sid_4_53> has title: Long Goodbye</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_413><sid_1_634><sid_2_358><sid_3_171><sid_4_27> has title: Love Is a Many-Splendored Thing</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_413><sid_1_634><sid_2_358><sid_3_171><sid_4_27> has title: The Last Picture Show</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_992><sid_1_839><sid_2_438><sid_3_205><sid_4_18> has title: Six Degrees of Separation</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_992><sid_1_839><sid_2_438><sid_3_205><sid_4_18> has title: Little Princess, The</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movi

In [14]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'genres' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><genres>The genres in movie <sid_0_38><sid_1_372><sid_2_340><sid_3_31><sid_4_122> are:Drama</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_38><sid_1_372><sid_2_340><sid_3_31><sid_4_122> are:Drama</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_405><sid_1_527><sid_2_22><sid_3_155><sid_4_53> are:Adventure,Children's,Fantasy</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_405><sid_1_527><sid_2_22><sid_3_155><sid_4_53> are:Drama</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_696><sid_1_308><sid_2_360><sid_3_251><sid_4_0> are:Comedy</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_696><sid_1_308><sid_2_360><sid_3_251><sid_4_0> are:Comedy,Drama</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The ge

In [15]:
n = 5

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'released' in true:

        in_pos = title_tokens.index('Ġin')
        prompt_tokens = title_tokens[:in_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><year>The movie <sid_0_38><sid_1_233><sid_2_500><sid_3_60><sid_4_52> was released in 1978</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_38><sid_1_233><sid_2_500><sid_3_60><sid_4_52> was released in 1999</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_1949><sid_1_650><sid_2_342><sid_3_110><sid_4_101> was released in 1978</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_1949><sid_1_650><sid_2_342><sid_3_110><sid_4_101> was released in 1996</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_1277><sid_1_452><sid_2_263><sid_3_50><sid_4_101> was released in 1997</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_1277><sid_1_452><sid_2_263><sid_3_50><sid_4_101> was released in 1997</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_405><sid_1_971><sid_2_464><sid_3

In [16]:
import re




def extract_genres_from_text(text: str) -> set[str]:
    """
    Достаём список жанров из сгенерированного текста.
    Ожидаем формат ... are:Comedy,Drama</genres> ...
    """
    # Берём кусок между 'are:' и '</genres>' (если есть)
    m = re.search(r'are:(.*?)(</genres>|<eos>|$)', text)
    if not m:
        return set()
    raw = m.group(1).strip()

    # Иногда модель может вставить пробелы/лишние символы
    raw = raw.replace(' ', '')
    if raw == '':
        return set()

    parts = [p for p in raw.split(',') if p]
    return set(parts)

def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

@torch.no_grad()
def predict_genres_from_prompt(prompt: str, model, tokenizer, device: str, max_new_tokens: int = 30) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # важно: для метрики лучше детерминированно
        pad_token_id=model.config.pad_token_id,
        eos_token_id=model.config.eos_token_id,
        bos_token_id=model.config.bos_token_id,
    )
    return tokenizer.decode(out[0])

def build_genre_prompt_from_true_example(example_text: str) -> str:
    """
    Из полного true-текста жанрового примера делаем prompt до 'are:' включительно.
    """
    idx = example_text.find('are:')
    if idx == -1:
        return example_text  # fallback
    return example_text[: idx + len('are:')]


# --- 2) Основная функция для mean Jaccard ---

def mean_jaccard_genres(ds_meta_genre, model, tokenizer, device: str, n: int = 200, seed: int = 42):
    rng = np.random.default_rng(seed)
    idxs = rng.integers(low=0, high=len(ds_meta_genre), size=n)

    scores = []
    empty_pred = 0

    for i in idxs:
        ids = ds_meta_genre[int(i)]['input_ids']
        true_text = tokenizer.decode(ids)

        prompt = build_genre_prompt_from_true_example(true_text)
        pred_text = predict_genres_from_prompt(prompt, model, tokenizer, device=device, max_new_tokens=30)

        true_set = extract_genres_from_text(true_text)
        pred_set = extract_genres_from_text(pred_text)

        if len(pred_set) == 0:
            empty_pred += 1

        scores.append(jaccard(true_set, pred_set))

    scores = np.array(scores, dtype=np.float32)
    return {
        'n': n,
        'mean_jaccard': float(scores.mean()),
        'std_jaccard': float(scores.std()),
        'empty_pred_frac': float(empty_pred / n),
        'scores': scores,  # если захочешь гистограмму/квантили
    }


# --- 3) Запуск ---

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

res = mean_jaccard_genres(ds_meta_genre, model, tokenizer, device=device, n=200, seed=0)
res

{'n': 200,
 'mean_jaccard': 0.3049166798591614,
 'std_jaccard': 0.4152837097644806,
 'empty_pred_frac': 0.0,
 'scores': array([1.        , 0.33333334, 0.5       , 0.        , 1.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.5       , 0.        ,
        1.        , 0.        , 0.        , 0.        , 1.        ,
        1.        , 0.        , 0.        , 0.33333334, 0.        ,
        1.        , 0.        , 0.        , 0.        , 0.33333334,
        0.        , 1.        , 0.5       , 0.5       , 1.        ,
        0.        , 1.        , 0.        , 1.        , 0.25      ,
        0.        , 0.        , 0.        , 0.        , 1.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.5       , 1.        , 0.33333334, 0.        , 0.        ,
        0.        , 0.        , 1.        , 0.        , 1.        ,
        0.        , 0.        , 0.        , 0.        , 0.       

In [18]:
n = 1

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if '<hist>' in true:

        in_pos = title_tokens.index('<hist>')
        prompt_tokens = title_tokens[:in_pos + 1]
        promt = tokenizer.convert_tokens_to_string(title_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('--------------------------------------------------')
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><user><gen_M><age_25><occ_7></user><hist><e><sid_0_696><sid_1_722><sid_2_54><sid_3_39><sid_4_30><rat_4></e><e><sid_0_992><sid_1_839><sid_2_483><sid_3_205><sid_4_49><rat_5></e><e><sid_0_540><sid_1_908><sid_2_183><sid_3_36><sid_4_30><rat_2></e><e><sid_0_739><sid_1_351><sid_2_485><sid_3_175><sid_4_51><rat_5></e><e><sid_0_540><sid_1_705><sid_2_425><sid_3_234><sid_4_121><rat_5></e><e><sid_0_405><sid_1_233><sid_2_445><sid_3_68><sid_4_75><rat_4></e><e><sid_0_405><sid_1_330><sid_2_172><sid_3_68><sid_4_101><rat_5></e><e><sid_0_739><sid_1_141><sid_2_360><sid_3_63><sid_4_96><rat_4></e><e><sid_0_739><sid_1_271><sid_2_18><sid_3_74><sid_4_13><rat_4></e><e><sid_0_38><sid_1_450><sid_2_208><sid_3_255><sid_4_96><rat_5></e><e><sid_0_992><sid_1_894><sid_2_365><sid_3_241><sid_4_79><rat_2></e><e><sid_0_38><sid_1_419><sid_2_147><sid_3_173><sid_4_51><rat_5></e><e><sid_0_696><sid_1_634><sid_2_17><sid_3_249><sid_4_100><rat_5></e><e><sid_0_739><sid_1_213><sid_2_63><sid_3_205><sid_4_13><r